# Sentiment Analysis of Customer Service Conversations

**Course:** DI 725 - Transformers and Attention-Based Deep Networks  
**Student Number:** 2786028  
**Model:** RoBERTa-base (fine-tuning)  
**Task:** 3-class sentiment classification (positive, negative, neutral)

This notebook implements a complete pipeline for sentiment analysis on customer service conversations using a fine-tuned RoBERTa model. Key design decisions:

- **Metadata prepend:** Categorical features (issue area, complexity, product, agent level) are prepended as text to the conversation
- **Head+Tail truncation:** First 128 + last 382 tokens to capture both problem statement and resolution
- **Class-weighted loss:** To handle severe class imbalance (positive class ~1.7%)
- **Experiment tracking:** All experiments logged to Weights & Biases

## 1. Setup & Imports

In [ ]:
!pip install transformers wandb scikit-learn -q

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

import wandb
import warnings
warnings.filterwarnings('ignore')

# --- Reproducibility ---
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
wandb.login()

## 2. Data Loading

In [ ]:
# Update this path if running on Google Colab
DATA_DIR = 'Assignment - 1 Dataset'

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train set: {train_df.shape}")
print(f"Test set:  {test_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
train_df.head(2)

## 3. Exploratory Data Analysis

In [ ]:
# Missing values
print("=== Missing Values (Train) ===")
print(train_df.isnull().sum())
print(f"\nTotal missing: {train_df.isnull().sum().sum()}")

print("\n=== Missing Values (Test) ===")
print(test_df.isnull().sum())
print(f"\nTotal missing: {test_df.isnull().sum().sum()}")

In [ ]:
# Sentiment class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Counts
sentiment_counts = train_df['customer_sentiment'].value_counts()
colors = {'negative': '#e74c3c', 'neutral': '#3498db', 'positive': '#2ecc71'}
bar_colors = [colors[s] for s in sentiment_counts.index]

axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=bar_colors)
axes[0].set_title('Sentiment Distribution (Counts)')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(zip(sentiment_counts.index, sentiment_counts.values)):
    axes[0].text(i, val + 200, str(val), ha='center', fontweight='bold')

# Percentages
sentiment_pct = (sentiment_counts / len(train_df) * 100).round(1)
axes[1].bar(sentiment_pct.index, sentiment_pct.values, color=bar_colors)
axes[1].set_title('Sentiment Distribution (%)')
axes[1].set_ylabel('Percentage')
for i, (idx, val) in enumerate(zip(sentiment_pct.index, sentiment_pct.values)):
    axes[1].text(i, val + 0.5, f'{val}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nClass imbalance ratio (max/min): {sentiment_counts.max() / sentiment_counts.min():.1f}x")

In [ ]:
# Conversation length analysis
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

train_df['word_count'] = train_df['conversation'].apply(lambda x: len(str(x).split()))
train_df['token_count'] = train_df['conversation'].apply(
    lambda x: len(tokenizer.encode(str(x), add_special_tokens=False))
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(train_df['word_count'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=train_df['word_count'].mean(), color='red', linestyle='--', label=f"Mean: {train_df['word_count'].mean():.0f}")
axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Word Count')
axes[0].legend()

axes[1].hist(train_df['token_count'], bins=50, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].axvline(x=512, color='red', linestyle='--', label='Max length (512)')
axes[1].axvline(x=train_df['token_count'].mean(), color='green', linestyle='--', label=f"Mean: {train_df['token_count'].mean():.0f}")
axes[1].set_title('RoBERTa Token Count Distribution')
axes[1].set_xlabel('Token Count')
axes[1].legend()

plt.tight_layout()
plt.show()

over_512 = (train_df['token_count'] > 512).mean() * 100
print(f"Conversations exceeding 512 tokens: {over_512:.1f}%")
print(f"Token count stats:\n{train_df['token_count'].describe().round(0)}")

In [ ]:
# Feature-sentiment correlation using Cramér's V
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    k = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * k)) if k > 0 else 0

cat_features = ['issue_area', 'issue_complexity', 'product_category', 'agent_experience_level']
correlations = {feat: cramers_v(train_df[feat], train_df['customer_sentiment']) for feat in cat_features}

plt.figure(figsize=(8, 4))
bars = plt.barh(list(correlations.keys()), list(correlations.values()), color='teal')
plt.xlabel("Cramér's V")
plt.title("Feature-Sentiment Correlation (Cramér's V)")
for bar, val in zip(bars, correlations.values()):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-tabulation: sentiment vs key features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.flatten(), cat_features):
    ct = pd.crosstab(train_df[feat], train_df['customer_sentiment'], normalize='index') * 100
    ct[['negative', 'neutral', 'positive']].plot(
        kind='barh', stacked=True, ax=ax,
        color=['#e74c3c', '#3498db', '#2ecc71']
    )
    ax.set_title(f'Sentiment by {feat}')
    ax.set_xlabel('Percentage')
    ax.legend(title='Sentiment', bbox_to_anchor=(1.0, 1.0))

plt.tight_layout()
plt.show()

### EDA Findings

1. **Class imbalance:** The dataset has severe imbalance — positive class is extremely underrepresented (~1.7%), while negative and neutral dominate. This justifies using **class-weighted loss** during training.

2. **Conversation length:** ~35% of conversations exceed the 512 token limit of RoBERTa. A **head+tail truncation** strategy (first 128 + last 382 tokens) is used to preserve both the problem statement and resolution/sentiment cues.

3. **Feature correlations:** Cramér's V analysis shows weak but non-zero correlations between categorical features (issue_area, issue_complexity, product_category, agent_experience_level) and sentiment. These features are **prepended as text metadata** to the conversation to provide additional context to the model.

4. **No missing values** detected in either train or test sets.